# Face-Car: treino do classificador de sonolência (CNN)

Este notebook treina a rede que olha **um quadro da câmera** e devolve 3 probabilidades, **nesta ordem**:

| saída | classe | significado |
|---|---|---|
| 0 | `alerta` | motorista acordado |
| 1 | `sonolento` | olhos fechando, cabeça caindo |
| 2 | `bocejo` | bocejando |

**Contrato com o projeto** (o que `entrada/classificador.py` espera):

- entrada: imagem **RGB**, redimensionada para `TAMANHO × TAMANHO` **sem recortar**, pixels **de 0 a 255** (a normalização é uma camada do modelo);
- saída: `softmax` com 3 valores que somam 1, na ordem acima, que é a mesma de `classificador.classes` em `config/analise.json`.

Junto do `.keras`, o notebook salva `cnn_sonolencia.classes.json` com a ordem das classes. O projeto lê esse arquivo e **se recusa a rodar** se a configuração estiver em outra ordem.

**Dados:** fotos e vídeos em `treinamento/dataset/<classe>/` (seção 1). Dos vídeos são tiradas algumas fotos por segundo. Hoje são os vídeos do **UTA-RLDD** (54 pessoas gravadas pela webcam ou celular, alertas e sonolentas) e as fotos próprias de 2 pessoas, organizados por `treinamento/organizar_dataset.py`.

**Como rodar:** ajuste a célula de **Configuração** e use *Executar tudo*. As células estão na ordem em que precisam rodar.

- **No computador:** abra este notebook no VS Code (ou Jupyter) com o kernel do `.venv` do projeto (`uv sync` instala tudo, inclusive o `ipykernel`). Sem GPU funciona, só que mais devagar.
- **No Colab:** suba a pasta `treinamento/dataset` para o Drive (padrão: `MyDrive/face_car/dataset`) e use *Ambiente de execução → Alterar o tipo → GPU (T4)*.

**Técnica usada (transfer learning):** a rede base (MobileNetV3Large, por padrão) já foi treinada pelo Google com 1,4 milhão de fotos (ImageNet) e sabe extrair bordas, texturas, olhos e rostos. Ela vira a "base" congelada, e só uma camada nova no fim aprende a separar as 3 classes. Depois, a parte final da base é liberada para um **ajuste fino**.

**Resultado:** `modelo/cnn_sonolencia.keras` e `modelo/cnn_sonolencia.classes.json`, na pasta que o projeto usa. O modelo anterior vai para `modelo/anteriores/`.

**Otimizações para acertar mais**, em relação ao FaceCar_CNN_V2 (cada uma é explicada na sua seção):

| o quê | antes (V2) | agora | seção |
|---|---|---|---|
| rede base | MobileNetV2 | MobileNetV3Large: mais precisa; ~29 ms por quadro no projeto, contra ~21 ms | 6 |
| resolução | 160 px | 224 px: 2× mais pixels nos olhos e na boca | 1 |
| aumento de dados | flip, rotação ±11°, zoom 5% | + deslocamento, zoom 20%, brilho/contraste, cinza (câmera IR), ruído | 6 |
| alvo do treino | rótulo "duro" | suavização de rótulos (0,1) | 7 |
| escolha da melhor época | erro da validação | F1 macro da validação (todas as classes contam) | 7 |
| taxa de aprendizado | fixa | cai sozinha quando o erro empaca | 7 |
| ajuste fino | — | 40% finais da base, com volta automática se piorar | 8 |
| classes desbalanceadas | — | pesos por classe | 7 |
| dados | fotos, divisão aleatória | fotos e vídeos, divisão por vídeo ou pessoa | 4 |

## 1. Configuração

Tudo o que muda de um treino para outro fica aqui.

O dataset fica em **`treinamento/dataset`**, com uma subpasta por classe. Dentro de cada uma pode haver **fotos e vídeos** misturados, inclusive em subpastas. **O nome de cada arquivo começa pela pessoa**, seguida de `_`:

```
treinamento/dataset/
├── alerta/      ← fotos (.jpg .png …) e/ou vídeos (.mp4 .mov …)
│   ├── rldd/      rldd01_0.mov, rldd02_0.mov …        (UTA-RLDD, vídeo "0")
│   └── proprio/   proprio1_acordado_0001.jpg …
├── sonolento/
│   ├── rldd/      rldd01_10.MOV …                      (UTA-RLDD, vídeo "10")
│   └── proprio/   proprio1_sonolento_0001.jpg, proprio1_olhos_fechados_0002.jpg …
├── bocejo/
│   └── proprio/   proprio1_bocejo_0004.jpg …
└── _fora_do_treino/   ← não está em CLASSES, então o treino ignora
    ├── pouco_alerta/  vídeos "5" do RLDD: num quadro isolado, iguais ao alerta
    └── duvidosas/     fotos cujo rosto contradiz a pasta (bocejo de boca fechada, piscada…)
```

Para acrescentar dados, ponha o arquivo na pasta da classe com o nome `pessoa_qualquercoisa` (ex.: `maria_bocejo_01.mp4`). Sempre o mesmo nome para a mesma pessoa.

- `CLASSES` liga **cada saída do modelo** à **subpasta** que tem os exemplos dela. A **ordem das chaves** é a ordem das saídas. Se a pasta tiver outro nome, mude só o valor, nunca a ordem.
- `QUADROS_POR_SEGUNDO` e `MAX_QUADROS_POR_VIDEO` dizem quantas fotos são tiradas de cada vídeo (seção 4).
- `BASE` e `TAMANHO` escolhem a rede base e a resolução (seção 6).
- `PROPORCOES` divide os dados em **treino** (a rede aprende), **validação** (decide quando parar) e **teste** (nota final, usado uma vez só, no fim).
- `REGEX_PESSOA`: veja a seção 4.

In [ ]:
# Saída do modelo → subpasta do dataset. A ORDEM é a das saídas e tem que ser a de
# classificador.classes em config/analise.json: alerta, sonolento, bocejo.
CLASSES = {
    "alerta": "alerta",
    "sonolento": "sonolento",
    "bocejo": "bocejo",
}

# None = automático. No computador: treinamento/dataset, treinamento/resultados (gráficos) e
# modelo/ na raiz do projeto (o .keras final). No Colab: a cópia de PASTA_DRIVE (suba para lá
# a pasta treinamento/dataset) e as pastas face_car/ e face_car/modelo/ no Drive.
PASTA_DATASET = None
PASTA_SAIDA = None
PASTA_MODELO = None
PASTA_DRIVE = "/content/drive/MyDrive/face_car/dataset"

NOME_MODELO = "FaceCar_CNN_V3"
ARQUIVO_MODELO = "cnn_sonolencia"   # gera cnn_sonolencia.keras e cnn_sonolencia.classes.json

QUADROS_POR_SEGUNDO = 2      # fotos tiradas de cada segundo de vídeo
MAX_QUADROS_POR_VIDEO = 60   # teto por vídeo (None = sem teto). Os vídeos do RLDD têm ~10 min: 60
                             # quadros = 1 a cada 10 s. Mais que isso afoga o bocejo, que só tem fotos

BASE = "MobileNetV3Large"   # ou "EfficientNetV2B0" (mais precisa, ~2x mais lenta) ou "MobileNetV2"
TAMANHO = 224                     # lado da imagem de entrada (o FaceCar_CNN_V2 usava 160)
LOTE = 32                         # imagens por passo de treino
PROPORCOES = (0.70, 0.15, 0.15)   # treino, validação, teste
SEED = 42                         # mesma semente → mesma divisão e mesmo sorteio

EPOCAS = 25                  # máximo da fase 1; o EarlyStopping costuma parar antes
AJUSTE_FINO = True           # fase 2: libera a parte final da rede base
EPOCAS_AJUSTE_FINO = 25
FRACAO_LIBERADA = 0.4        # fração final das camadas da base que volta a aprender na fase 2
SUAVIZACAO_ROTULOS = 0.1     # seção 7
PESOS_BASE = "imagenet"      # pesos pré-treinados da rede base

# Separação por pessoa (seção 4). Grupo 1 da regex = quem aparece, tirado do NOME do arquivo
# (da foto ou do vídeo): o que vem antes do primeiro "_". "rldd07_10.mp4" → "rldd07",
# "proprio1_bocejo_0004.jpg" → "proprio1". None = separa por vídeo/foto.
REGEX_PESSOA = r"^([^_]+)_"
FOTOS_POR_BLOCO = 15   # classe com pessoas de menos para dividir por pessoa (o bocejo): seção 4.3

CACHE_NA_MEMORIA = True      # guarda as imagens já decodificadas na RAM (desligue se faltar memória)

## 2. Bibliotecas

Só o que é usado: **TensorFlow/Keras** (rede e leitura das imagens), **OpenCV** (quadros dos vídeos), **scikit-learn** (divisão dos dados e métricas), **NumPy** e **matplotlib**.

`keras.utils.set_random_seed` fixa de uma vez as sementes do Python, do NumPy e do TensorFlow. A GPU ainda pode dar diferenças minúsculas entre execuções, mas a divisão dos dados sai sempre igual.

In [ ]:
import importlib.util
import json
import os
import re
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import date
from pathlib import Path

import cv2
import keras
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from keras import layers
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

keras.utils.set_random_seed(SEED)
EM_COLAB = importlib.util.find_spec("google.colab") is not None
NOMES = list(CLASSES)   # ["alerta", "sonolento", "bocejo"]: índice = saída do modelo

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| OpenCV", cv2.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "nenhuma (o treino vai ser lento)")
print("Colab:", EM_COLAB)

## 3. Onde estão os dados

- **No computador** (VS Code ou Jupyter, com o `.venv` do projeto): usa `treinamento/dataset` direto, põe gráficos em `treinamento/resultados/` e o modelo final em `modelo/`, na raiz do projeto. Tudo isso fica fora do git.
- **No Colab:** monta o Google Drive e copia `PASTA_DRIVE` para o disco do Colab. Ler milhares de arquivos direto do Drive é muito lento, porque cada leitura passa pela rede. Os gráficos vão para a pasta `face_car` do Drive e o modelo para `face_car/modelo`.

Os quadros tirados dos vídeos ficam em `quadros/`, ao lado do dataset (seção 4).

In [ ]:
if EM_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if PASTA_DATASET is None:
        PASTA_DATASET = "/content/dataset"
        inicio = time.time()
        shutil.copytree(PASTA_DRIVE, PASTA_DATASET, dirs_exist_ok=True)
        print(f"Cópia do Drive concluída em {(time.time() - inicio) / 60:.1f} min")
    PASTA_SAIDA = PASTA_SAIDA or str(Path(PASTA_DRIVE).parent)
    PASTA_MODELO = PASTA_MODELO or str(Path(PASTA_DRIVE).parent / "modelo")
else:
    # O notebook roda com a pasta dele (treinamento/) ou a raiz do projeto como atual.
    treinamento = Path.cwd() if Path.cwd().name == "treinamento" else Path.cwd() / "treinamento"
    PASTA_DATASET = PASTA_DATASET or str(treinamento / "dataset")
    PASTA_SAIDA = PASTA_SAIDA or str(treinamento / "resultados")
    PASTA_MODELO = PASTA_MODELO or str(treinamento.parent / "modelo")

PASTA_QUADROS = Path(PASTA_DATASET).parent / "quadros"
Path(PASTA_SAIDA).mkdir(parents=True, exist_ok=True)
Path(PASTA_MODELO).mkdir(parents=True, exist_ok=True)
print("Dataset:   ", PASTA_DATASET)
print("Quadros:   ", PASTA_QUADROS)
print("Resultados:", PASTA_SAIDA)
print("Modelo:    ", PASTA_MODELO)

## 4. Fotos, vídeos e a divisão em treino, validação e teste

### 4.1 O que há em cada classe

A célula confere se as subpastas de `CLASSES` existem e conta fotos e vídeos. Arquivos de outro tipo são listados e ignorados, nunca descartados em silêncio. As fotos precisam estar em JPG, PNG ou BMP, os formatos que o TensorFlow lê. Se houver `.webp` ou `.heic`, converta antes.

In [ ]:
FOTOS = {".jpg", ".jpeg", ".png", ".bmp"}
VIDEOS = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v", ".3gp"}

raiz = Path(PASTA_DATASET)
if not raiz.is_dir():
    raise FileNotFoundError(f"A pasta do dataset não existe: {raiz}")
subpastas = sorted(p.name for p in raiz.iterdir() if p.is_dir())
faltando = [pasta for pasta in CLASSES.values() if pasta not in subpastas]
if faltando:
    raise FileNotFoundError(f"Subpastas {faltando} não existem em {raiz}. Existem: {subpastas}. "
                            "Crie-as ou ajuste CLASSES na configuração.")
ignoradas = sorted(set(subpastas) - set(CLASSES.values()))
if ignoradas:
    print("Subpastas ignoradas (não estão em CLASSES):", ignoradas)

fotos, videos, outros = {}, {}, []
for nome, pasta in CLASSES.items():
    arquivos = sorted(f for f in (raiz / pasta).rglob("*") if f.is_file() and not f.name.startswith("."))
    fotos[nome] = [f for f in arquivos if f.suffix.lower() in FOTOS]
    videos[nome] = [f for f in arquivos if f.suffix.lower() in VIDEOS]
    outros += [f for f in arquivos if f.suffix.lower() not in FOTOS | VIDEOS]
    print(f"{nome:<10} pasta '{pasta}': {len(fotos[nome])} fotos, {len(videos[nome])} vídeos")
if outros:
    print(f"\nIgnorados {len(outros)} arquivos de tipo não suportado, por exemplo:",
          [str(f.relative_to(raiz)) for f in outros[:5]])

### 4.2 Quadros dos vídeos

A rede aprende com **fotos**, então cada vídeo vira uma sequência delas. Não se usa todo quadro: um vídeo a 30 quadros por segundo tem 30 fotos quase iguais a cada segundo. Isso só deixa o treino mais lento e faz o vídeo pesar demais. A célula tira `QUADROS_POR_SEGUNDO` fotos por segundo, espaçadas por igual, e no máximo `MAX_QUADROS_POR_VIDEO` por vídeo.

- Os quadros são salvos em `quadros/<classe>/<vídeo>@<configuração>/`, com o lado maior reduzido a 640 px (a rede usa 160 px, então não se perde nada).
- A extração acontece **uma vez só**. Na próxima execução, os quadros prontos são reaproveitados. Se `QUADROS_POR_SEGUNDO` ou o teto mudarem, o nome da pasta muda e os quadros são extraídos de novo. Uma extração interrompida também é refeita.
- O OpenCV entrega os quadros já desvirados, mesmo de vídeo de celular gravado em pé.
- Vários vídeos são processados ao mesmo tempo, um por núcleo do processador.

In [ ]:
LADO_MAXIMO = 640
CONFIG_QUADROS = f"{QUADROS_POR_SEGUNDO}qps-max{MAX_QUADROS_POR_VIDEO or 'livre'}"


def pasta_dos_quadros(video, nome):
    relativo = video.relative_to(raiz / CLASSES[nome])
    return PASTA_QUADROS / CLASSES[nome] / f"{'__'.join(relativo.parts)}@{CONFIG_QUADROS}"


def extrair_quadros(video, destino):
    """Salva os quadros escolhidos do vídeo em `destino`; devolve os arquivos, em ordem."""
    marca = destino / ".completo"
    if marca.exists():
        return sorted(destino.glob("q*.jpg"))
    shutil.rmtree(destino, ignore_errors=True)
    destino.mkdir(parents=True)

    captura = cv2.VideoCapture(str(video))
    if not captura.isOpened():
        print(f"  Aviso: não consegui abrir {video.name}; vídeo ignorado")
        return []
    fps = captura.get(cv2.CAP_PROP_FPS) or 30
    total = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
    passo = max(1, round(fps / QUADROS_POR_SEGUNDO))
    escolhidos = np.arange(0, max(total, 1), passo)
    if MAX_QUADROS_POR_VIDEO and len(escolhidos) > MAX_QUADROS_POR_VIDEO:
        escolhidos = escolhidos[np.linspace(0, len(escolhidos) - 1, MAX_QUADROS_POR_VIDEO).astype(int)]
    escolhidos = set(escolhidos.tolist()) if total > 0 else None   # sem total: só o passo

    salvos, indice = [], 0
    while captura.grab():   # grab() só avança; retrieve() decodifica, e só dos escolhidos
        if (indice % passo == 0) if escolhidos is None else (indice in escolhidos):
            ok, quadro = captura.retrieve()
            if ok:
                escala = LADO_MAXIMO / max(quadro.shape[:2])
                if escala < 1:
                    quadro = cv2.resize(quadro, None, fx=escala, fy=escala,
                                        interpolation=cv2.INTER_AREA)
                arquivo = destino / f"q{indice:07d}.jpg"
                cv2.imwrite(str(arquivo), quadro, [cv2.IMWRITE_JPEG_QUALITY, 95])
                salvos.append(arquivo)
        indice += 1
        if MAX_QUADROS_POR_VIDEO and len(salvos) >= MAX_QUADROS_POR_VIDEO \
                or escolhidos and indice > max(escolhidos):
            break
    captura.release()
    marca.write_text(json.dumps({"video": str(video), "fps": fps, "quadros": len(salvos)}))
    return salvos


tarefas = [(nome, video) for nome in NOMES for video in videos[nome]]
inicio = time.time()
with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
    quadros = list(executor.map(lambda t: extrair_quadros(t[1], pasta_dos_quadros(t[1], t[0])),
                                tarefas))
quadros_do_video = {video: arquivos for (_, video), arquivos in zip(tarefas, quadros)}
if tarefas:
    print(f"{len(tarefas)} vídeos → {sum(map(len, quadros))} quadros "
          f"em {time.time() - inicio:.0f} s ({CONFIG_QUADROS})")

### 4.3 Juntar tudo e dividir

Cada exemplo (foto ou quadro de vídeo) recebe um **grupo**, que diz de onde ele veio:

- quadro de vídeo → o **vídeo**;
- foto → a própria foto;
- com `REGEX_PESSOA` (o padrão) → a **pessoa**, tirada do nome da foto ou do vídeo.

**Um grupo nunca é dividido**: todos os quadros de um vídeo caem no mesmo conjunto. Isso evita o **vazamento de dados**. Dois quadros vizinhos de um vídeo são quase a mesma foto. Se um caísse no treino e o outro no teste, a rede acertaria o teste só por "lembrar" do vídeo, e a nota ficaria perto de 100% sem valer para vídeos novos. Pelo mesmo motivo, dividir **por pessoa** é ainda melhor, porque mede se a rede funciona com **gente que ela nunca viu**.

A divisão é **estratificada por grupo** (`StratifiedGroupKFold`): respeita os grupos e, dentro disso, tenta manter em cada conjunto a mesma proporção de alerta, sonolento e bocejo. Com poucos vídeos ou pessoas, as proporções ficam aproximadas. Por isso, grave **vários vídeos curtos por classe** em vez de um longo: cada classe precisa de pelo menos uns 7 grupos para aparecer nos três conjuntos.

**Classe com pessoas de menos.** Para separar 15% das pessoas, uma classe precisa de pelo menos 7 pessoas. Senão, algum conjunto fica sem ela, e a validação não consegue medir essa classe. Hoje é o caso do bocejo, que só tem fotos de 2 pessoas. Nessa classe, as fotos de cada pessoa são repartidas em blocos de `FOTOS_POR_BLOCO` fotos seguidas, e cada bloco é um grupo. O bocejo aparece nos três conjuntos, mas a mesma pessoa bocejando fica no treino e no teste, então **a nota do bocejo no teste é otimista**. As outras classes continuam divididas por pessoa. Com bocejos de 7 pessoas ou mais, isso deixa de acontecer sozinho.

Se uma classe tiver muito mais exemplos que outra, a rede tende a "chutar" a maior. Na seção 7, cada classe ganha um **peso** inverso ao seu tamanho para compensar.

In [ ]:
def pessoa(arquivo):
    achou = re.search(REGEX_PESSOA, Path(arquivo).name)
    if not achou:
        raise ValueError(f"REGEX_PESSOA não achou a pessoa em '{Path(arquivo).name}'")
    return achou.group(1)


caminhos, rotulos, grupos, de_video = [], [], [], []
for indice, nome in enumerate(NOMES):
    origens = [(foto, [foto], False) for foto in fotos[nome]]
    origens += [(video, quadros_do_video[video], True) for video in videos[nome]]
    for origem, arquivos, eh_video in origens:
        grupo = pessoa(origem) if REGEX_PESSOA else str(origem)
        caminhos += [str(a) for a in arquivos]
        rotulos += [indice] * len(arquivos)
        grupos += [grupo] * len(arquivos)
        de_video += [eh_video] * len(arquivos)
caminhos, rotulos = np.array(caminhos), np.array(rotulos)
grupos, de_video = np.array(grupos, dtype=object), np.array(de_video)

# Classe com pessoas de menos para a divisão por pessoa: blocos de fotos seguidas viram os grupos.
_, fracao_val, fracao_teste = PROPORCOES
GRUPOS_MINIMOS = max(round(1 / fracao_teste), round((1 - fracao_teste) / fracao_val))
classes_em_blocos = []
for indice, nome in enumerate(NOMES):
    da_classe = np.flatnonzero(rotulos == indice)
    pessoas = sorted(set(grupos[da_classe]))
    if not REGEX_PESSOA or not 0 < len(pessoas) < GRUPOS_MINIMOS:
        continue
    classes_em_blocos.append(nome)
    for quem in pessoas:
        dela = da_classe[grupos[da_classe] == quem]
        for k, i in enumerate(dela[np.argsort(caminhos[dela])]):
            grupos[i] = f"{quem}/{nome}/bloco{k // FOTOS_POR_BLOCO:02d}"
    print(f"Atenção: '{nome}' só tem {len(pessoas)} pessoas ({', '.join(pessoas)}); para dividir por "
          f"pessoa são precisas {GRUPOS_MINIMOS}. As fotos dela foram repartidas em blocos de "
          f"{FOTOS_POR_BLOCO}: a nota de '{nome}' no teste fica otimista.\n")

contagem = np.bincount(rotulos, minlength=len(NOMES))
print(f"{'':<10}{'fotos':>7}{'quadros':>9}{'total':>7}{'grupos':>8}")
for indice, nome in enumerate(NOMES):
    da_classe = rotulos == indice
    print(f"{nome:<10}{(da_classe & ~de_video).sum():>7}{(da_classe & de_video).sum():>9}"
          f"{da_classe.sum():>7}{len(set(grupos[da_classe])):>8}")
if contagem.min() == 0:
    vazias = [NOMES[i] for i in np.flatnonzero(contagem == 0)]
    raise ValueError(f"Classe sem nenhum exemplo: {vazias}. Coloque fotos ou vídeos na pasta dela.")
if contagem.max() / contagem.min() > 1.5:
    print(f"\nAtenção: classes desbalanceadas (a maior tem {contagem.max() / contagem.min():.1f}x "
          "a menor). Os pesos por classe da seção 7 compensam.")

In [ ]:
_, FRACAO_VAL, FRACAO_TESTE = PROPORCOES


def separar(indices, fracao):
    """Separa ~`fracao` dos grupos de `indices`, estratificando pela classe."""
    partes = max(2, round(1 / fracao))
    if len(set(grupos[indices])) < partes:
        raise ValueError(f"Só há {len(set(grupos[indices]))} grupos (vídeos/fotos/pessoas) para "
                         f"separar {fracao:.0%}; são precisos pelo menos {partes}. Adicione dados "
                         "ou mude PROPORCOES.")
    corte = StratifiedGroupKFold(n_splits=partes, shuffle=True, random_state=SEED)
    fica, sai = next(corte.split(indices, rotulos[indices], grupos[indices]))
    return indices[fica], indices[sai]


resto, idx_teste = separar(np.arange(len(caminhos)), FRACAO_TESTE)
idx_treino, idx_val = separar(resto, FRACAO_VAL / (1 - FRACAO_TESTE))

conjuntos = {"treino": idx_treino, "validação": idx_val, "teste": idx_teste}
for (a, ia), (b, ib) in [(("treino", idx_treino), ("validação", idx_val)),
                         (("treino", idx_treino), ("teste", idx_teste)),
                         (("validação", idx_val), ("teste", idx_teste))]:
    assert not set(grupos[ia]) & set(grupos[ib]), f"um grupo caiu em {a} e em {b}"

print(f"{'':<10}" + "".join(f"{n:>11}" for n in NOMES) + f"{'total':>8}{'grupos':>8}")
for nome, idx in conjuntos.items():
    por_classe = np.bincount(rotulos[idx], minlength=len(NOMES))
    print(f"{nome:<10}" + "".join(f"{n:>11}" for n in por_classe)
          + f"{len(idx):>8}{len(set(grupos[idx])):>8}")
    if por_classe.min() == 0:
        print(f"  Atenção: '{nome}' ficou sem alguma classe; adicione mais vídeos/fotos dela.")

## 5. Leitura das imagens (`tf.data`)

Cada imagem passa por estes passos:

1. lê o arquivo e decodifica em **RGB** (3 canais, mesmo se a foto for cinza);
2. redimensiona para `TAMANHO × TAMANHO` **sem recortar**, como `preparar()` faz no projeto com `cv2.resize`. Treino e uso precisam preparar a imagem do mesmo jeito;
3. arredonda para inteiros de 0 a 255, como a câmera entrega. Assim o cache ocupa 4× menos memória;
4. `cache`: depois da 1ª época, as imagens já vêm prontas da memória;
5. só no treino: `shuffle` embaralha a cada época, e um **chuvisco** leve (`RUIDO`) imita o ruído de webcam em pouca luz. Ele é sorteado de novo a cada época;
6. o rótulo vira **one-hot**: `sonolento` = `[0, 1, 0]`. É o formato que a suavização de rótulos e a métrica F1 da seção 7 precisam;
7. `batch` junta em lotes de `LOTE`, e `prefetch` prepara o próximo lote enquanto a GPU treina o atual.

Arquivo corrompido não derruba o treino: é pulado com um aviso (`ignore_errors`).

In [ ]:
def carregar_imagem(caminho, rotulo):
    imagem = tf.io.decode_image(tf.io.read_file(caminho), channels=3, expand_animations=False)
    imagem = tf.image.resize(imagem, (TAMANHO, TAMANHO))   # bilinear, sem recortar
    imagem = tf.cast(tf.clip_by_value(tf.round(imagem), 0, 255), tf.uint8)
    return imagem, rotulo


def para_o_modelo(imagem, rotulo):
    # o modelo recebe float32 de 0 a 255; o rótulo vai em one-hot
    return tf.cast(imagem, tf.float32), tf.one_hot(rotulo, len(NOMES))


def com_ruido(imagem, rotulo):
    # chuvisco de webcam em pouca luz: desvio de RUIDO níveis (de 0 a 255), sorteado a cada época
    ruido = tf.random.normal(tf.shape(imagem), stddev=RUIDO)
    return tf.clip_by_value(imagem + ruido, 0, 255), rotulo


def montar(indices, treinar):
    ds = tf.data.Dataset.from_tensor_slices((caminhos[indices], rotulos[indices]))
    ds = ds.map(carregar_imagem, num_parallel_calls=tf.data.AUTOTUNE).ignore_errors(log_warning=True)
    if CACHE_NA_MEMORIA:
        ds = ds.cache()
    if treinar:
        ds = ds.shuffle(len(indices), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(para_o_modelo, num_parallel_calls=tf.data.AUTOTUNE)
    if treinar and RUIDO:
        ds = ds.map(com_ruido, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(LOTE).prefetch(tf.data.AUTOTUNE)


RUIDO = 6.0
treino = montar(idx_treino, treinar=True)
validacao = montar(idx_val, treinar=False)
teste = montar(idx_teste, treinar=False)

imagens, rots = next(iter(treino))
plt.figure(figsize=(9, 9))
for i in range(min(9, len(imagens))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(imagens[i].numpy().astype("uint8"))
    plt.title(NOMES[int(np.argmax(rots[i]))])
    plt.axis("off")
plt.suptitle("Amostras do treino (confira se o rótulo bate com a foto)")
plt.show()

## 6. O modelo

Uma imagem atravessa estas camadas (formas para a MobileNetV3Large a 224 px):

| camada | o que faz | forma |
|---|---|---|
| `imagem_motorista` | entrada RGB, pixels de 0 a 255 | 224×224×3 |
| `aumento_dados` | **só no treino**: cria variações da foto (abaixo) | 224×224×3 |
| *base* (`BASE`) | normaliza os pixels e extrai características visuais | 7×7×960 |
| `GlobalAveragePooling2D` | média de cada mapa de características: resume a imagem em um vetor | 960 |
| `Dropout(0.3)` | **só no treino**: desliga 30% das entradas ao acaso, para não decorar | 960 |
| `classificacao` | `Dense` + `softmax`: 3 probabilidades que somam 1 | 3 |

**Rede base.** A padrão é a **MobileNetV3Large**, sucessora da MobileNetV2 do modelo V2. No ImageNet ela acerta 75,6%, contra 71,3% da V2. Na análise, num notebook sem GPU, ela leva ~29 ms por quadro a 224 px, contra ~21 ms do V2 a 160 px: segue acima de 30 quadros por segundo. A `EfficientNetV2B0` acerta mais (78,7%), mas é cerca de 2× mais lenta. A MobileNetV3 e a EfficientNetV2 já trazem a normalização dos pixels dentro delas. A MobileNetV2 precisa da camada `Rescaling`, que o código põe sozinho.

**Aumento de dados.** A cada época a rede vê versões diferentes de cada foto e aprende o que **não** importa para a resposta:

- espelhar, deslocar até 8%, girar até ±14° e aproximar ou afastar até 20%: o rosto não fica sempre no mesmo lugar nem na mesma distância da câmera;
- brilho e contraste ±25%: dia, noite, luz do painel;
- 15% das fotos em **cinza**: parecido com a câmera infravermelha;
- chuvisco de webcam: entra na leitura dos dados (seção 5).

No uso real (`predict`), todas essas camadas ficam desligadas.

**Por que `base(x, training=False)`?** A base tem camadas de *BatchNormalization*, que guardam médias aprendidas no ImageNet. Com `training=False`, elas continuam usando essas médias mesmo no ajuste fino. Sem isso, um lote pequeno bagunçaria as médias e estragaria a base.

In [ ]:
BASES = {   # nome: (construtor, a base já normaliza os pixels de 0 a 255?)
    "MobileNetV3Large": (keras.applications.MobileNetV3Large, True),
    "EfficientNetV2B0": (keras.applications.EfficientNetV2B0, True),
    "MobileNetV2": (keras.applications.MobileNetV2, False),
}
construtor, normaliza_sozinha = BASES[BASE]

aumento = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.08, 0.08, fill_mode="reflect"),
    layers.RandomRotation(0.04, fill_mode="reflect"),            # 0,04 × 360° = ±14,4°
    layers.RandomZoom((-0.2, 0.2), fill_mode="reflect"),
    layers.RandomBrightness(0.25, value_range=(0, 255)),
    layers.RandomContrast(0.25),
    layers.RandomGrayscale(0.15),
], name="aumento_dados")

base = construtor(weights=PESOS_BASE, include_top=False, input_shape=(TAMANHO, TAMANHO, 3))
base.trainable = False

entrada = keras.Input((TAMANHO, TAMANHO, 3), name="imagem_motorista")
x = aumento(entrada)
if not normaliza_sozinha:
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)   # 0..255 → -1..1
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
saida = layers.Dense(len(NOMES), activation="softmax", name="classificacao")(x)
modelo = keras.Model(entrada, saida, name=NOME_MODELO)

modelo.summary()

## 7. Treino, fase 1: só a camada nova aprende

- **Adam** com taxa de aprendizado 0,001: o algoritmo que ajusta os pesos.
- **Entropia cruzada com suavização de rótulos** (`SUAVIZACAO_ROTULOS = 0.1`): a "nota de erro" pune a rede quando ela dá probabilidade baixa à classe certa. Com a suavização, o alvo de uma foto sonolenta deixa de ser `[0, 1, 0]` e vira `[0,03, 0,93, 0,03]`. A rede para de buscar 100% de certeza, que é onde ela começa a decorar detalhes das fotos de treino. Assim ela erra menos em fotos novas, e as probabilidades ficam mais realistas, o que importa porque o projeto tira a **média** delas.
- **Pesos por classe** (`class_weight`): errar uma classe com poucas fotos custa mais. Assim a rede não ganha nota "chutando" a classe maior.
- **A melhor época é escolhida pela F1 macro da validação**, e não pelo erro. A F1 macro é a média do equilíbrio entre precision e recall **de cada classe**: uma rede que ignora o bocejo fica com nota baixa, mesmo que acerte muito o resto.
- **EarlyStopping**: se a F1 da validação não melhorar em algumas épocas, o treino para e volta aos pesos da melhor época.
- **ReduceLROnPlateau**: se o erro da validação empacar por 2 épocas, a taxa de aprendizado cai para 30%. Passos menores ajudam a achar um ponto melhor.
- **Uma linha por época** (`verbose=2`) aqui e em `resultados/treino_fase1.csv`, gravado a cada época.

In [ ]:
pesos = compute_class_weight("balanced", classes=np.arange(len(NOMES)), y=rotulos[idx_treino])
PESOS_CLASSE = dict(enumerate(pesos))
print("Pesos por classe:", {NOMES[i]: round(float(p), 2) for i, p in PESOS_CLASSE.items()})

PERDA = keras.losses.CategoricalCrossentropy(label_smoothing=SUAVIZACAO_ROTULOS)


def metricas():
    return ["accuracy", keras.metrics.F1Score(average="macro", name="f1")]


def callbacks(paciencia, fase):
    # treino_fase1.csv / treino_fase2.csv: uma linha por época, gravada na hora. Para acompanhar
    # um treino rodando fora do editor: tail -f treinamento/resultados/treino_fase1.csv
    return [keras.callbacks.EarlyStopping(monitor="val_f1", mode="max", patience=paciencia,
                                          restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2,
                                              min_lr=1e-7),
            keras.callbacks.CSVLogger(Path(PASTA_SAIDA) / f"treino_fase{fase}.csv")]


modelo.compile(optimizer=keras.optimizers.Adam(1e-3), loss=PERDA, metrics=metricas())
inicio = time.time()
hist_fase1 = modelo.fit(treino, validation_data=validacao, epochs=EPOCAS, verbose=2,
                        class_weight=PESOS_CLASSE, callbacks=callbacks(paciencia=5, fase=1))
print(f"Fase 1: {(time.time() - inicio) / 60:.1f} min")
historicos = [hist_fase1]

### Curvas de aprendizado

Como ler:

- **as curvas de treino e de validação melhorando juntas**: está aprendendo;
- **treino continua melhorando, validação piora**: *overfitting* (decoreba). O EarlyStopping corta aqui;
- **validação melhor que treino**: normal, porque o aumento de dados, o Dropout e a suavização só agem no treino. Se a diferença for exagerada, desconfie de vazamento (seção 4).

In [ ]:
def mostrar_curvas(historicos, arquivo=None):
    juntar = lambda chave: sum((h.history[chave] for h in historicos), [])
    epocas = np.arange(1, len(juntar("loss")) + 1)
    fig, eixos = plt.subplots(1, 3, figsize=(16, 4))
    for eixo, chave, titulo in [(eixos[0], "loss", "Erro (loss)"),
                                (eixos[1], "accuracy", "Acurácia"),
                                (eixos[2], "f1", "F1 macro (escolhe a melhor época)")]:
        eixo.plot(epocas, juntar(chave), label="treino")
        eixo.plot(epocas, juntar("val_" + chave), label="validação")
        if len(historicos) > 1:
            eixo.axvline(len(historicos[0].history["loss"]) + 0.5, color="gray", linestyle="--",
                         label="início do ajuste fino")
        eixo.set_title(titulo)
        eixo.set_xlabel("época")
        eixo.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        eixo.legend()
    if arquivo:
        fig.savefig(arquivo, dpi=150, bbox_inches="tight")
    plt.show()


mostrar_curvas(historicos)

## 8. Treino, fase 2: ajuste fino (opcional)

Na fase 1 a base ficou congelada. Agora a parte final dela (`FRACAO_LIBERADA`, 40% das camadas) volta a aprender. São as camadas que reconhecem padrões mais específicos, como "olho meio fechado" ou "boca bem aberta". As primeiras (bordas, texturas) servem para qualquer foto e continuam congeladas.

- A taxa de aprendizado começa em 3e-5, 33 vezes menor que na fase 1. Se fosse alta, a rede apagaria em poucos passos o que aprendeu no ImageNet. O `compile` é obrigatório depois de mudar o que é treinável.
- Só vale a pena **depois** da fase 1. Se a camada nova ainda estiver aleatória, os erros grandes dela estragam a base.
- **Rede de segurança:** os pesos da fase 1 ficam guardados. Se, no fim, o ajuste fino tiver uma F1 de validação **pior**, o modelo volta para a fase 1. O ajuste fino nunca piora o resultado.

In [ ]:
if AJUSTE_FINO:
    guardado_fase1 = Path(PASTA_SAIDA) / "fase1.weights.h5"
    modelo.save_weights(guardado_fase1)
    f1_fase1 = modelo.evaluate(validacao, verbose=0, return_dict=True)["f1"]

    liberadas = int(len(base.layers) * FRACAO_LIBERADA)
    base.trainable = True
    for camada in base.layers[:-liberadas]:
        camada.trainable = False
    for camada in base.layers:
        if isinstance(camada, layers.BatchNormalization):
            camada.trainable = False   # as médias do ImageNet ficam como estão
    print(f"Camadas treináveis na base: {sum(c.trainable for c in base.layers)} de {len(base.layers)}")

    modelo.compile(optimizer=keras.optimizers.Adam(3e-5), loss=PERDA, metrics=metricas())
    inicio = time.time()
    hist_fase2 = modelo.fit(treino, validation_data=validacao, epochs=EPOCAS_AJUSTE_FINO, verbose=2,
                            class_weight=PESOS_CLASSE, callbacks=callbacks(paciencia=5, fase=2))
    print(f"Fase 2: {(time.time() - inicio) / 60:.1f} min")
    historicos.append(hist_fase2)

    f1_fase2 = modelo.evaluate(validacao, verbose=0, return_dict=True)["f1"]
    print(f"F1 macro na validação: fase 1 = {f1_fase1:.3f}, ajuste fino = {f1_fase2:.3f}")
    if f1_fase2 < f1_fase1:
        modelo.load_weights(guardado_fase1)
        print("O ajuste fino piorou: o modelo voltou para os pesos da fase 1.")
    guardado_fase1.unlink()

mostrar_curvas(historicos, Path(PASTA_SAIDA) / "curvas_treino.png")

## 9. Avaliação no teste (a nota honesta)

Agora sim o teste é usado, e só esta vez. Para cada imagem, a classe prevista é a de maior probabilidade (`argmax`).

- **precision** de sonolento: das vezes que a rede disse "sonolento", quantas estavam certas (muitos alarmes falsos = precision baixa);
- **recall** de sonolento: dos motoristas realmente sonolentos, quantos ela pegou. **É a métrica mais importante aqui**, porque deixar passar alguém dormindo é muito pior que um alarme falso;
- **f1**: equilíbrio entre as duas;
- **matriz de confusão**: linha = classe real, coluna = classe prevista. A diagonal são os acertos, e o resto mostra *qual* classe é confundida com *qual*.

Imagens e rótulos são lidos **no mesmo laço**, para a previsão de cada foto ficar junto do rótulo dela.

In [ ]:
y_real, y_prob = [], []
for imagens, rots in teste:
    y_prob.append(modelo.predict_on_batch(imagens))
    y_real.append(rots.numpy().argmax(axis=1))
y_real, y_prob = np.concatenate(y_real), np.concatenate(y_prob)
y_prev = y_prob.argmax(axis=1)

relatorio = classification_report(y_real, y_prev, labels=range(len(NOMES)), target_names=NOMES,
                                  digits=3, zero_division=0, output_dict=True)
print(classification_report(y_real, y_prev, labels=range(len(NOMES)), target_names=NOMES,
                            digits=3, zero_division=0))

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
matriz = confusion_matrix(y_real, y_prev, labels=range(len(NOMES)))
ConfusionMatrixDisplay(matriz, display_labels=NOMES).plot(ax=eixos[0], cmap="Blues",
                                                          values_format="d", colorbar=False)
eixos[0].set_title("Quantidade")
ConfusionMatrixDisplay(confusion_matrix(y_real, y_prev, labels=range(len(NOMES)), normalize="true"),
                       display_labels=NOMES).plot(ax=eixos[1], cmap="Blues", values_format=".0%",
                                                  colorbar=False)
eixos[1].set_title("Por linha (recall de cada classe)")
for eixo in eixos:
    eixo.set_xlabel("previsto")
    eixo.set_ylabel("real")
fig.suptitle(f"Matriz de confusão no teste - {NOME_MODELO} ({BASE}, {TAMANHO} px)")
fig.savefig(Path(PASTA_SAIDA) / "matriz_confusao.png", dpi=150, bbox_inches="tight")
plt.show()

### E se o alarme disparar antes do 50%?

Com o `argmax`, a rede só diz "sonolento" quando essa é a maior probabilidade. Num sistema de segurança, pode valer a pena alertar mais cedo, por exemplo com P(sonolento) ≥ 30%: pega mais motoristas sonolentos (recall ↑) ao custo de mais alarmes falsos (precision ↓). A tabela mostra essa troca no teste e ajuda a escolher o limiar do projeto.

In [ ]:
SONOLENTO = NOMES.index("sonolento")
real = y_real == SONOLENTO
print(f"{'limiar':>7} {'recall':>8} {'precision':>10} {'alarmes':>8}")
for limiar in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    alarme = y_prob[:, SONOLENTO] >= limiar
    recall = (alarme & real).sum() / max(real.sum(), 1)
    precisao = (alarme & real).sum() / max(alarme.sum(), 1)
    print(f"{limiar:>7.0%} {recall:>8.1%} {precisao:>10.1%} {alarme.sum():>8}")

## 10. Salvar o modelo na pasta `modelo/`

São salvos dois arquivos em `modelo/`, a pasta que o projeto usa (`classificador.modelo` em `config/analise.json`):

- `cnn_sonolencia.keras`: arquitetura, pesos e pré-processamento, tudo junto;
- `cnn_sonolencia.classes.json`: a ordem das saídas, a base, o tamanho de entrada e as notas no teste. O projeto lê este arquivo e confere com `classificador.classes`.

Para não perder nada:

1. o modelo novo é salvo primeiro com outro nome e **recarregado**, e a célula confere se ele prevê exatamente o mesmo que o da memória;
2. só então o modelo anterior (se houver) vai para `modelo/anteriores/<data e hora>/`;
3. e o novo assume o nome `cnn_sonolencia.keras`.

Para voltar ao modelo anterior, é só mover os dois arquivos de `anteriores/` de volta para `modelo/`.

In [ ]:
pasta_modelo = Path(PASTA_MODELO)
caminho_modelo = pasta_modelo / f"{ARQUIVO_MODELO}.keras"
caminho_classes = pasta_modelo / f"{ARQUIVO_MODELO}.classes.json"
provisorio = pasta_modelo / f"{ARQUIVO_MODELO}.novo.keras"

modelo.save(provisorio)
recarregado = keras.models.load_model(provisorio, compile=False)
lote = next(iter(teste))[0]
np.testing.assert_allclose(recarregado.predict_on_batch(lote), modelo.predict_on_batch(lote),
                           atol=1e-5)

anteriores = [c for c in (caminho_modelo, caminho_classes) if c.exists()]
if anteriores:
    guarda = pasta_modelo / "anteriores" / time.strftime("%Y-%m-%d_%H%M%S")
    guarda.mkdir(parents=True)
    for arquivo in anteriores:
        shutil.move(arquivo, guarda / arquivo.name)
    print("Modelo anterior guardado em", guarda)
provisorio.rename(caminho_modelo)

caminho_classes.write_text(json.dumps({
    "modelo": NOME_MODELO,
    "classes": NOMES,
    "pastas": CLASSES,
    "base": BASE,
    "entrada": [TAMANHO, TAMANHO, 3],
    "pixels": "RGB de 0 a 255, quadro inteiro redimensionado sem recorte",
    "divisao": "por pessoa" if REGEX_PESSOA else "por vídeo/foto",
    "classes_em_blocos": classes_em_blocos,   # divididas em blocos de fotos, não por pessoa
    "quadros_dos_videos": CONFIG_QUADROS,
    "exemplos": {"fotos": int((~de_video).sum()), "quadros_de_video": int(de_video.sum())},
    "teste": {
        "imagens": int(len(y_real)),
        "acuracia": round(relatorio["accuracy"], 4),
        "f1_macro": round(relatorio["macro avg"]["f1-score"], 4),
        "recall": {n: round(relatorio[n]["recall"], 4) for n in NOMES},
        "precision": {n: round(relatorio[n]["precision"], 4) for n in NOMES},
    },
    "treinado_em": date.today().isoformat(),
}, ensure_ascii=False, indent=2), encoding="utf-8")

print("Salvo e conferido:")
print(" ", caminho_modelo)
print(" ", caminho_classes)

if EM_COLAB:
    from google.colab import files

    files.download(str(caminho_modelo))
    files.download(str(caminho_classes))

## 11. Usar no projeto

- **No computador:** pronto. A análise (`python -m entrada`) e o simulador já leem `modelo/cnn_sonolencia.keras`, e o tamanho de entrada é lido do próprio modelo.
- **No Colab:** os dois arquivos ficam em `face_car/modelo/` no Drive e são baixados no fim. Coloque-os em `modelo/` no projeto.

Se `classificador.classes` em `config/analise.json` não for `["alerta", "sonolento", "bocejo"]`, o projeto avisa a diferença e não carrega o modelo.

**O que mais aumentaria o acerto:**

- **Dados parecidos com o uso.** Vídeos gravados **na mesma webcam e posição** em que o sistema vai rodar, de **várias pessoas**, com e sem óculos, em luzes diferentes. Nenhuma técnica compensa a diferença entre fotos de treino e imagens de uso: é por isso que o V2 dava "sonolento" para um rosto acordado.
- **Recortar o rosto.** A rede recebe o **quadro inteiro**, e os olhos ocupam poucos pixels dele. Recortar o rosto (no treino e no uso, com o MediaPipe que o projeto já tem) deve ser o maior ganho que ainda falta, mas exige mudar também a análise e o simulador.
- **Bocejos de mais pessoas.** O bocejo só tem fotos de 2 pessoas (o RLDD não tem essa classe). Com a divisão por pessoa, uma delas pode ir inteira para o teste e a rede aprende bocejo com uma pessoa só. Vídeos curtos de bocejo de várias pessoas, ou o dataset público YawDD (pedido aos autores), são o que mais falta.